# AI Impact Dimensions by Company AI Strategy

Classifies companies into **Augmentation**, **Automation**, **Both**, or **Other** (not in classification file),
then generates AI Impact Dimension time-series charts for each category.

Based on: Raisch & Krakowski (2021) Classification from `500.xlsx`

In [ ]:
# =============================================================================
# CONFIGURATION & IMPORTS
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
import json
import ast
import gc
import glob
from typing import List, Dict, Any
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

INPUT_FOLDER = '../data/extracted_skills_production'
SKILLS_COLUMN = 'skills'
DATE_COLUMN = 'post_date'
RCID_COLUMN = 'rcid'
CLASSIFICATION_FILE = '500.xlsx'

# --- Helper function (reused from analyze_extracted_skills_multifile.ipynb) ---
def parse_skills_column(value) -> List[str]:
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        return [str(s) for s in value if s is not None]
    if isinstance(value, str):
        if value in ('[]', '', 'nan', 'None'):
            return []
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            pass
    return []

# --- Load AI Impact Lookup ---
AI_IMPACT_LOOKUP_PATH = Path('../skillner/data/ksao_ai_impact_lookup.json')
if AI_IMPACT_LOOKUP_PATH.exists():
    with open(AI_IMPACT_LOOKUP_PATH) as f:
        AI_IMPACT_LOOKUP = json.load(f)
    print(f"Loaded AI Impact lookup with {len(AI_IMPACT_LOOKUP):,} KSAOs")
else:
    AI_IMPACT_LOOKUP = {}
    print(f"WARNING: AI Impact lookup not found at {AI_IMPACT_LOOKUP_PATH}")

# --- Find parquet files ---
parquet_files = sorted(glob.glob(f'{INPUT_FOLDER}/*.parquet'))
print(f"Found {len(parquet_files)} parquet files")

In [ ]:
# =============================================================================
# LOAD & PROCESS COMPANY CLASSIFICATIONS FROM 500.xlsx
# =============================================================================

import openpyxl

wb = openpyxl.load_workbook(CLASSIFICATION_FILE, read_only=True, data_only=True)
ws = wb['Sheet1']

# Read headers
headers = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
print("Columns:", headers)

# Find column indices
rcid_col = headers.index('rcid')
class_col = headers.index('Raisch & Krakowski (2021) Classification')
name_col = headers.index('Company name')

VALID_CLASSES = {'Augmentation', 'Automation', 'Both'}

# Collect all (rcid, classification) pairs
company_rows = []  # (rcid, company_name, classification)
for row in ws.iter_rows(min_row=2, values_only=True):
    rcid_val = row[rcid_col]
    classification = row[class_col]
    company_name = row[name_col]
    
    # Skip invalid rcids
    if rcid_val is None or str(rcid_val).strip() in ('#N/A', '', 'N/A', 'nan'):
        continue
    # Skip non-standard classifications
    if classification not in VALID_CLASSES:
        continue
    
    try:
        rcid_int = int(float(str(rcid_val)))
    except (ValueError, TypeError):
        continue
    
    company_rows.append((rcid_int, company_name, classification))

wb.close()

# Apply priority rule: if ANY row for an RCID says 'Both', the company is 'Both'
# Otherwise, if it has both 'Augmentation' and 'Automation' rows, it's 'Both'
rcid_classes = defaultdict(set)  # rcid -> set of classifications
rcid_names = {}  # rcid -> company name
for rcid_int, company_name, classification in company_rows:
    rcid_classes[rcid_int].add(classification)
    rcid_names[rcid_int] = company_name

rcid_to_category = {}  # rcid -> final category
for rcid_int, classes in rcid_classes.items():
    if 'Both' in classes:
        rcid_to_category[rcid_int] = 'Both'
    elif 'Augmentation' in classes and 'Automation' in classes:
        rcid_to_category[rcid_int] = 'Both'
    elif 'Automation' in classes:
        rcid_to_category[rcid_int] = 'Automation'
    else:
        rcid_to_category[rcid_int] = 'Augmentation'

# Summary
category_counts = Counter(rcid_to_category.values())
print(f"\nCompany classifications (by unique RCID):")
print(f"  Total classified companies: {len(rcid_to_category)}")
for cat in ['Augmentation', 'Automation', 'Both']:
    print(f"  {cat}: {category_counts.get(cat, 0)} companies")

# Convert rcid keys to strings for matching with parquet data
rcid_to_category_str = {str(k): v for k, v in rcid_to_category.items()}
print(f"\nReady to match against parquet files.")

In [ ]:
# =============================================================================
# STREAM PARQUET FILES: Compute Quarterly AI Impact by Category
# =============================================================================

# category -> quarter -> {dimension -> count, '__jd_count__' -> count}
CATEGORIES = ['Augmentation', 'Automation', 'Both', 'Other']
DIM_NAMES = ['Replaced by AI', 'Augmented by AI', 'Building/Managing AI',
             'Resistant to AI', 'Transformed by AI']

def make_empty_quarter_dict():
    d = {dim: 0 for dim in DIM_NAMES}
    d['__jd_count__'] = 0
    return d

cat_quarterly = {cat: defaultdict(make_empty_quarter_dict) for cat in CATEGORIES}

print("Scanning parquet files for quarterly AI Impact by company category...")

for i, filepath in enumerate(parquet_files):
    try:
        df_part = pd.read_parquet(filepath, columns=[RCID_COLUMN, DATE_COLUMN, SKILLS_COLUMN])
    except Exception as e:
        print(f"  Skipping {filepath}: {e}")
        continue
    
    df_part[DATE_COLUMN] = pd.to_datetime(df_part[DATE_COLUMN], errors='coerce')
    df_part['quarter'] = df_part[DATE_COLUMN].dt.to_period('Q')
    
    for _, row in df_part.iterrows():
        q = row['quarter']
        if pd.isna(q):
            continue
        q_str = str(q)
        
        # Determine category
        rcid_val = row[RCID_COLUMN]
        rcid_str = str(int(float(rcid_val))) if pd.notna(rcid_val) else ''
        category = rcid_to_category_str.get(rcid_str, 'Other')
        
        cat_quarterly[category][q_str]['__jd_count__'] += 1
        
        skills_list = parse_skills_column(row.get(SKILLS_COLUMN))
        for skill in skills_list:
            skill_lower = skill.lower().strip()
            if skill_lower in AI_IMPACT_LOOKUP:
                dim = AI_IMPACT_LOOKUP[skill_lower]['ai_impact_dimension']
                if dim in cat_quarterly[category][q_str]:
                    cat_quarterly[category][q_str][dim] += 1
    
    del df_part
    gc.collect()
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(parquet_files)}] processed")

print(f"\nDone. Building DataFrames...")

# Build one DataFrame per category
ai_col_names = [f'ai_{d.replace(" ", "_").replace("/", "_").lower()}' for d in DIM_NAMES]

quarterly_dfs = {}
for cat in CATEGORIES:
    quarters_sorted = sorted(cat_quarterly[cat].keys())
    rows_list = []
    for q in quarters_sorted:
        jd_count = cat_quarterly[cat][q]['__jd_count__']
        if jd_count == 0:
            continue
        row_data = {'year_quarter': q}
        for dim, col_name in zip(DIM_NAMES, ai_col_names):
            row_data[col_name] = cat_quarterly[cat][q][dim] / jd_count
        rows_list.append(row_data)
    quarterly_dfs[cat] = pd.DataFrame(rows_list)
    n_jds = sum(cat_quarterly[cat][q]['__jd_count__'] for q in cat_quarterly[cat])
    print(f"  {cat}: {len(quarterly_dfs[cat])} quarters, {n_jds:,} total JDs")

print("\nReady for visualization.")

In [ ]:
# =============================================================================
# VISUALIZATION: 4 Individual Charts (one per category)
# =============================================================================

dim_mapping = {
    'ai_replaced_by_ai': ('Replaced by AI', '#e74c3c'),
    'ai_augmented_by_ai': ('Augmented by AI', '#3498db'),
    'ai_building_managing_ai': ('Building/Managing AI', '#9b59b6'),
    'ai_resistant_to_ai': ('Resistant to AI', '#27ae60'),
    'ai_transformed_by_ai': ('Transformed by AI', '#f39c12')
}

def plot_ai_impact_over_time(quarterly_ai, title, category_counts_dict=None):
    """Generate stacked area + line chart for one category."""
    if quarterly_ai is None or len(quarterly_ai) == 0:
        print(f"No data for: {title}")
        return
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    x_labels = [str(q) for q in quarterly_ai['year_quarter']]
    
    # 1. Stacked area chart
    ax1 = axes[0]
    bottom = np.zeros(len(quarterly_ai))
    for col, (label, color) in dim_mapping.items():
        if col in quarterly_ai.columns:
            values = quarterly_ai[col].values
            ax1.fill_between(range(len(x_labels)), bottom, bottom + values,
                             label=label, color=color, alpha=0.7)
            bottom = bottom + values
    ax1.set_xticks(range(0, len(x_labels), max(1, len(x_labels)//15)))
    ax1.set_xticklabels([x_labels[i] for i in range(0, len(x_labels), max(1, len(x_labels)//15))],
                        rotation=45, ha='right')
    ax1.set_ylabel('Average Skills per JD')
    ax1.set_title(f'{title} — Composition Over Time', fontsize=12)
    ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    ax1.grid(axis='y', alpha=0.3)
    
    # 2. Line chart with markers
    ax2 = axes[1]
    for col, (label, color) in dim_mapping.items():
        if col in quarterly_ai.columns:
            ax2.plot(range(len(x_labels)), quarterly_ai[col].values,
                     label=label, color=color, linewidth=2, marker='o', markersize=3)
    ax2.set_xticks(range(0, len(x_labels), max(1, len(x_labels)//15)))
    ax2.set_xticklabels([x_labels[i] for i in range(0, len(x_labels), max(1, len(x_labels)//15))],
                        rotation=45, ha='right')
    ax2.set_xlabel('Quarter')
    ax2.set_ylabel('Average Skills per JD')
    ax2.set_title(f'{title} — Trends Over Time', fontsize=12)
    ax2.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print trend summary
    print(f"\n{title} — Change from first to last quarter:")
    print("-" * 60)
    for col, (label, _) in dim_mapping.items():
        if col in quarterly_ai.columns and len(quarterly_ai) > 1:
            first_val = quarterly_ai[col].iloc[1] if len(quarterly_ai) > 1 else quarterly_ai[col].iloc[0]
            last_val = quarterly_ai[col].iloc[-1]
            change = ((last_val - first_val) / first_val * 100) if first_val > 0 else 0
            trend = '↑' if change > 0 else '↓' if change < 0 else '→'
            print(f"  {label:<25}: {first_val:.2f} → {last_val:.2f} ({trend} {abs(change):.1f}%)")
    print()

# Count companies per category
cat_company_counts = Counter(rcid_to_category.values())

# Generate 4 charts
for cat in CATEGORIES:
    n_companies = cat_company_counts.get(cat, 0)
    if cat == 'Other':
        label = f'Other Companies (not in classification)'
    else:
        label = f'{cat} Companies (N={n_companies})'
    plot_ai_impact_over_time(quarterly_dfs.get(cat), f'AI Impact Dimensions — {label}')

In [ ]:
# =============================================================================
# COMBINED 2x2 COMPARISON CHART
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes_flat = axes.flatten()

cat_company_counts = Counter(rcid_to_category.values())

for idx, cat in enumerate(CATEGORIES):
    ax = axes_flat[idx]
    qdf = quarterly_dfs.get(cat)
    
    if qdf is None or len(qdf) == 0:
        ax.text(0.5, 0.5, f'No data for {cat}', ha='center', va='center', fontsize=14)
        ax.set_title(cat)
        continue
    
    x_labels = [str(q) for q in qdf['year_quarter']]
    
    for col, (label, color) in dim_mapping.items():
        if col in qdf.columns:
            ax.plot(range(len(x_labels)), qdf[col].values,
                    label=label, color=color, linewidth=2, marker='o', markersize=2)
    
    tick_step = max(1, len(x_labels) // 10)
    ax.set_xticks(range(0, len(x_labels), tick_step))
    ax.set_xticklabels([x_labels[i] for i in range(0, len(x_labels), tick_step)],
                       rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Avg Skills per JD', fontsize=9)
    ax.grid(alpha=0.3)
    
    n_companies = cat_company_counts.get(cat, 0)
    if cat == 'Other':
        ax.set_title(f'Other (not classified)', fontsize=11)
    else:
        ax.set_title(f'{cat} (N={n_companies})', fontsize=11)

# Shared legend
handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('AI Impact Dimensions Over Time — By Company AI Strategy',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()